# Формирую df numbeo country index

In [1]:
from os import getenv
from datetime import date

import country_converter as coco
import pandas as pd

#### From Cost of Living Index by Country 2022

In [2]:
df_col_index = pd.read_html('https://www.numbeo.com/cost-of-living/rankings_by_country.jsp')[1]
df_col_index.drop('Rank', axis=1, inplace=True)
df_col_index.columns = list(map(lambda x: x.removesuffix(' Index').lower().replace(' ', '_'), df_col_index.columns))

#### From Quality of Life Index by Country

In [3]:
df_qol_index = pd.read_html('https://www.numbeo.com/quality-of-life/rankings_by_country.jsp')[1]
df_qol_index.drop(['Rank', 'Cost of Living Index', 'Safety Index'], axis=1, inplace=True)
df_qol_index.columns = list(map(lambda x: x.removesuffix(' Index').lower().replace(' ', '_'), df_qol_index.columns))

#### From Average Monthly Net Salary by Country

In [4]:
df_salary_index = pd.read_html('https://www.numbeo.com/cost-of-living/country_price_rankings?itemId=105')[1]
df_salary_index.drop([0, 2], axis=1, inplace=True)
df_salary_index[3] = df_salary_index[3].apply(lambda row: float(row.rstrip('\xa0$').replace(',', '')))
df_salary_index.rename(columns={1: 'country', 3: 'avg_salary_usd'}, inplace=True)

#### From Crime Index by Country

In [5]:
df_safety_index = pd.read_html('https://www.numbeo.com/crime/rankings_by_country.jsp')[1]
df_safety_index = df_safety_index[['Country', 'Safety Index']]
df_safety_index.columns = ['country', 'safety']

#### Merge 4 Country Tables

In [6]:
df_country_index = df_col_index.merge(df_qol_index, how='left', on='country')
df_country_index = df_country_index.merge(df_salary_index, how='left', on='country')
df_country_index = df_country_index.merge(df_safety_index, how='left', on='country')
# добавляю столбцы когда и кем добавлены данные в таблицу
df_country_index.loc[:, ('sys_updated_date', 'sys_updated_by')] = (date.today(), getenv('NB_USER'))

Removing Kosovo

In [7]:
df_country_index = df_country_index.drop(
    df_country_index.loc[df_country_index['country'] == 'Kosovo (Disputed Territory)'].index)

In [8]:
df_country_index['country_code'] = df_country_index.apply(lambda row: coco.convert(names=row.country, to='ISO3'),
                                                          axis=1)
df_country_index = df_country_index.reindex(columns=['country_code'] + list(df_country_index.columns[1:-1]))
df_country_index.head(2)

,country_code,cost_of_living,rent,cost_of_living_plus_rent,groceries,restaurant_price,local_purchasing_power,quality_of_life,purchasing_power,health_care,property_price_to_income_ratio,traffic_commute_time,pollution,climate,avg_salary_usd,safety,sys_updated_date,sys_updated_by
0,BMU,135.8,108.2,123.5,143.4,147.5,101.3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-03-12,analyst_k2
1,CYM,115.6,76.1,97.9,124.0,101.5,149.6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,69.5,2026-03-12,analyst_k2


In [9]:
df_country_index.head(20)

,country_code,cost_of_living,rent,cost_of_living_plus_rent,groceries,restaurant_price,local_purchasing_power,quality_of_life,purchasing_power,health_care,property_price_to_income_ratio,traffic_commute_time,pollution,climate,avg_salary_usd,safety,sys_updated_date,sys_updated_by
0,BMU,135.8,108.2,123.5,143.4,147.5,101.3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-03-12,analyst_k2
1,CYM,115.6,76.1,97.9,124.0,101.5,149.6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,69.5,2026-03-12,analyst_k2
2,VIR,111.3,46.8,82.5,127.4,94.8,89.8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.8,2026-03-12,analyst_k2
3,CHE,110.7,51.5,84.3,109.7,111.3,170.6,206.2,176.1,71.2,11.4,34.5,24.3,80.7,7613.01,72.6,2026-03-12,analyst_k2
4,SLB,102.3,19.5,65.4,64.8,46.0,12.6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-03-12,analyst_k2
5,BHS,98.8,50.2,77.1,100.9,109.0,57.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,43.1,2026-03-12,analyst_k2
6,ISL,97.2,49.5,75.9,104.5,104.9,113.7,195.8,117.6,69.1,7.7,20.7,17.0,68.8,NaN,74.5,2026-03-12,analyst_k2
7,JEY,88.7,52.4,72.5,79.0,102.3,103.4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-03-12,analyst_k2
8,SGP,87.7,73.1,81.2,77.3,55.5,105.5,158.1,110.6,71.9,22.1,41.0,32.2,57.5,4251.44,77.5,2026-03-12,analyst_k2
9,NOR,83.7,29.2,59.4,85.4,88.6,124.7,195.4,128.0,75.8,8.0,28.0,19.0,70.6,3994.20,66.7,2026-03-12,analyst_k2


# Загрузка в БД

In [11]:
from sqlalchemy import create_engine, text
engine = create_engine(getenv("SQLALCHEMY_RELOHELPER_URL"))

In [12]:
df_country_index.to_sql('numbeo_country_indices', engine, if_exists='append', index=False)

stmt6 = "ALTER TABLE numbeo_country_indices ADD CONSTRAINT PK_country_code2 PRIMARY KEY (country_code)"
stmt7 = "ALTER TABLE numbeo_country_indices ADD CONSTRAINT FK_country_code FOREIGN KEY (country_code) REFERENCES public.countries (country_code)"

with engine.connect() as connection:
    connection.execute(text(stmt6))
    connection.execute(text(stmt7))
    connection.commit()